In [ ]:
!git clone https://github.com/rm-a0/3d-nca --quiet
%cd /kaggle/working/3d-nca
!pip install pyvista

In [ ]:
import os, sys, random
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from torch import Tensor
from typing import List

sys.path.insert(0, '/kaggle/working/3d-nca')
 
from nca3d.core.cell       import CellConfig
from nca3d.core.grid       import Grid3D, GridConfig
from nca3d.core.perception import PerceptionConfig
from nca3d.core.update     import UpdateConfig
from nca3d.core.nca_model  import NCAModel, NCAConfig
from nca3d.server.logger   import NCALogger
from nca3d.viz.volume_mpl  import show_volume_rgba_mpl, show_state_rgba_mpl
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
GRID         = 32
PHASE_EPOCHS = 3000          # 3 phases x 3000 = 9000 total
POOL_SIZE    = 32
DRIVE_DIR    = "/kaggle/working/nca_morph"
os.makedirs(DRIVE_DIR, exist_ok=True)
 
config = {
    "cell": {
        "hidden_channels":  16,
        "visible_channels": 4,
        "alive_threshold":  0.1,
        "task_channels":    3,
        "pos_channels":     3,
    },
    "perception": {
        "kernel_radius":  1,
        "channel_groups": 5,
    },
    "update": {
        "hidden_dim":        96,
        "stochastic_update": False,
        "fire_rate":         0.5,
    },
    "grid": {
        "size": (GRID, GRID, GRID),
    },
    "training": {
        "learning_rate": 2e-3,
        "batch_size":    4,
    },
}
 
N_TASKS = config["cell"]["task_channels"]
VIS     = config["cell"]["visible_channels"]
TC      = config["cell"]["task_channels"]
PC      = config["cell"]["pos_channels"]
TOTAL_EPOCHS = PHASE_EPOCHS * N_TASKS   # 9000

In [ ]:
def _rgba(mask, r, g, b):
    out = np.zeros((*mask.shape, 4), dtype=np.float32)
    out[mask, :3] = [r, g, b]
    out[mask,  3] = 1.0
    return out
 
def make_tetrahedron(grid=GRID, pad=3):
    n = grid - 2 * pad
    i, j, k = np.mgrid[0:n, 0:n, 0:n]
    x = (i + 0.5) / n * 2 - 1
    y = (j + 0.5) / n * 2 - 1
    z = (k + 0.5) / n * 2 - 1
    s = 0.85
    mask = (x+y+z <= s) & (-x-y+z <= s) & (-x+y-z <= s) & (x-y-z <= s)
    full = np.zeros((grid, grid, grid), bool)
    full[pad:pad+n, pad:pad+n, pad:pad+n] = mask
    return _rgba(full, 0.35, 0.55, 1.0)
 
def make_cube(grid=GRID, pad=5):
    full = np.zeros((grid, grid, grid), bool)
    full[pad:grid-pad, pad:grid-pad, pad:grid-pad] = True
    return _rgba(full, 1.0, 0.6, 0.2)
 
def make_sphere(grid=GRID):
    c, r = grid / 2.0, grid / 2.0 - 2.0
    i, j, k = np.mgrid[0:grid, 0:grid, 0:grid]
    mask = (i-c)**2 + (j-c)**2 + (k-c)**2 <= r**2
    return _rgba(mask, 0.2, 0.9, 0.4)
 
TARGETS    = [make_tetrahedron(), make_cube(), make_sphere()]
TASK_NAMES = ["Tetrahedron", "Cube", "Sphere"]
 
print("Target voxel counts:")
for name, t in zip(TASK_NAMES, TARGETS):
    print(f"  {name:12s}: {int((t[..., 3] > 0).sum()):>5} voxels")

In [ ]:
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "figure.facecolor": "white"})
 
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), subplot_kw={"projection": "3d"})
for col, (rgba, name) in enumerate(zip(TARGETS, TASK_NAMES)):
    n = show_volume_rgba_mpl(rgba, ax=axes[col], surface_only=True,
                             threshold=0.15, point_size=14,
                             view_angle=(28, 40), show=False, title="")
    axes[col].set_title(f"{name}\n({n} surface voxels)", fontsize=9)
fig.suptitle("Training targets -- surface voxels only", fontsize=11)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/targets.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cell_cfg = CellConfig(**config["cell"])
perc_cfg = PerceptionConfig(**config["perception"])
upd_cfg  = UpdateConfig(**config["update"])
grid_cfg = GridConfig(**config["grid"])
 
model = Grid3D(cell_cfg, perc_cfg, upd_cfg, grid_cfg).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel -- {total_params:,} trainable params")
print(f"  hidden={cell_cfg.hidden_channels}  pos={PC}  task={TC}  "
      f"vis={VIS}  total={cell_cfg.total_channels}  "
      f"perception_groups={perc_cfg.channel_groups}")
 
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=1e-5,
)
# Scheduler runs for the full 9000 epochs.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-5,
)

In [ ]:
def step_range(epoch):
    """Global curriculum: step count ramps from (8,16) to (32,64) over the
    first PHASE_EPOCHS epochs, then stays at maximum."""
    t = min(epoch / PHASE_EPOCHS, 1.0)
    return max(int(8 + t * 24), 4), max(int(16 + t * 48), 5)
 
def prep_target(rgba):
    """(D,H,W,4) -> (1,4,D,H,W) on device."""
    t = np.transpose(rgba, (3, 0, 1, 2)).astype(np.float32)
    return torch.from_numpy(t).unsqueeze(0).to(device)
 
# Pre-compute target tensors once.
targets_t = [prep_target(t) for t in TARGETS]
 
def new_seed(task_id):
    """Single alive centre cell with task one-hot."""
    return model.seed_center(1, device, torch.tensor([task_id], device=device))
 
def relabel(state, task_id):
    """Overwrite task-channel embedding across all cells in-place (cloned copy).
 
    Channel layout (innermost = index 0):
      [hidden(16) | pos(3) | task(3) | visible(4)]
    Task slice: state[:, -(VIS+TC) : -VIS, ...]
    Positional channels are left alone; Grid3D.step re-injects them every step.
    """
    s = state.clone()
    one_hot = F.one_hot(
        torch.tensor([task_id], device=s.device), num_classes=TC
    ).float()
    s[:, -(VIS + TC) : -VIS, ...] = (
        one_hot.view(1, TC, 1, 1, 1).expand(-1, -1, *s.shape[2:])
    )
    return s
 
def expand_pool(pool, pool_tasks, from_task, to_task, n_convert):
    """Convert the n_convert best from_task pool slots to to_task.
 
    "Best" = lowest MSE against the from_task target (most converged).
    These states are the ideal morphing inputs for to_task because they
    are the closest representations of what from_task should look like.
    The remaining from_task slots stay as from_task for continued training.
    """
    from_indices = [i for i, t in enumerate(pool_tasks) if t == from_task]
    tgt = targets_t[from_task]
    with torch.no_grad():
        scored = sorted(
            [(F.mse_loss(pool[i][:, -VIS:], tgt[:, -VIS:]).item(), i)
             for i in from_indices]
        )  # ascending: best first
    to_convert = [i for _, i in scored[:n_convert]]
    for idx in to_convert:
        pool[idx] = relabel(pool[idx], to_task)
        pool_tasks[idx] = to_task
    return pool, pool_tasks
 
def train_step(pool, pool_tasks, epoch):
    """One gradient step on a random batch sampled across all active tasks."""
    bs      = min(len(pool), config["training"]["batch_size"])
    lo, hi  = step_range(epoch)
    n_steps = random.randint(lo, hi)
 
    indices    = random.sample(range(len(pool)), bs)
    batch      = torch.cat([pool[i] for i in indices], dim=0)
    task_ids_b = [pool_tasks[i] for i in indices]
    tgt_batch  = torch.cat([targets_t[t] for t in task_ids_b], dim=0)
 
    # Worst-sample reset.
    # Resets preserve the morphing chain: task_N inputs come from task_(N-1) pool.
    with torch.no_grad():
        per_loss = [
            F.mse_loss(batch[i:i+1, -VIS:], targets_t[task_ids_b[i]][:, -VIS:]).item()
            for i in range(bs)
        ]
        worst_b    = int(np.argmax(per_loss))
        worst_task = task_ids_b[worst_b]
 
        if worst_task == 0:
            batch[worst_b:worst_b+1] = new_seed(0)
        else:
            # Reseed from a random pool member of the parent task, relabeled.
            parent_task    = worst_task - 1
            parent_indices = [i for i, t in enumerate(pool_tasks) if t == parent_task]
            if parent_indices:
                src = random.choice(parent_indices)
                batch[worst_b:worst_b+1] = relabel(pool[src], worst_task)
            else:
                batch[worst_b:worst_b+1] = new_seed(worst_task)
 
    # Forward pass.
    optimizer.zero_grad()
    state    = model(batch, steps=n_steps)
    pred_vis = state[:, -VIS:]
    tgt_vis  = tgt_batch[:, -VIS:]
    tgt_a    = tgt_vis[:, -1:]
    tgt_mask = (tgt_a > cell_cfg.alive_threshold).float()
 
    loss_alpha    = F.mse_loss(pred_vis[:, -1:], tgt_a)
    color_diff    = (pred_vis[:, :-1] - tgt_vis[:, :-1]) ** 2 * tgt_mask
    loss_color    = color_diff.sum() / tgt_mask.sum().clamp(min=1.0)
    loss_overflow = ((pred_vis[:, -1:] * (1.0 - tgt_mask)) ** 2).mean()
    loss = 4.0 * loss_alpha + 1.0 * loss_color + 2.0 * loss_overflow
 
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
 
    with torch.no_grad():
        for j, idx in enumerate(indices):
            pool[idx] = state[j:j+1].detach()
 
    return {
        "loss_total":    loss.item(),
        "loss_alpha":    loss_alpha.item(),
        "loss_color":    loss_color.item(),
        "loss_overflow": loss_overflow.item(),
    }, pool

In [ ]:
logger = NCALogger(base_dir=DRIVE_DIR, checkpoint_interval=500)
logger.log_meta(config)
print(f"Run dir: {logger.run_dir}")
 

In [ ]:
pool       = [new_seed(0) for _ in range(POOL_SIZE)]
pool_tasks = [0] * POOL_SIZE
 
print(f"\nTraining starts -- {TOTAL_EPOCHS} epochs, {N_TASKS} tasks")
print(f"Epoch 1 to {PHASE_EPOCHS}: task 0 only (seed -> tetrahedron)")
print(f"Epoch {PHASE_EPOCHS+1} to {2*PHASE_EPOCHS}: tasks 0+1 jointly")
print(f"Epoch {2*PHASE_EPOCHS+1} to {TOTAL_EPOCHS}: all 3 tasks jointly")
print()
 
for epoch in range(1, TOTAL_EPOCHS + 1):
 
    # Pool expansion at phase boundaries.
    if epoch == PHASE_EPOCHS + 1:
        pool, pool_tasks = expand_pool(pool, pool_tasks, 0, 1, n_convert=16)
        dist = dict(Counter(pool_tasks))
        print(f"  Epoch {epoch}: pool expanded -> task distribution {dist}")
 
    elif epoch == PHASE_EPOCHS * 2 + 1:
        pool, pool_tasks = expand_pool(pool, pool_tasks, 1, 2, n_convert=8)
        dist = dict(Counter(pool_tasks))
        print(f"  Epoch {epoch}: pool expanded -> task distribution {dist}")
 
    metrics, pool = train_step(pool, pool_tasks, epoch)
 
    # Determine phase label for logger (for loss.csv phase column).
    if epoch <= PHASE_EPOCHS:
        phase_label = "0"
    elif epoch <= PHASE_EPOCHS * 2:
        phase_label = "0+1"
    else:
        phase_label = "0+1+2"
 
    logger.log_epoch(
        epoch         = epoch,
        loss_alpha    = metrics["loss_alpha"],
        loss_color    = metrics["loss_color"],
        loss_overflow = metrics["loss_overflow"],
        loss_total    = metrics["loss_total"],
        phase         = phase_label,
        model         = model,
        is_final      = (epoch == TOTAL_EPOCHS),
    )
 
    if epoch % 500 == 0:
        dist = dict(Counter(pool_tasks))
        print(f"  [{epoch:>5}/{TOTAL_EPOCHS}]  "
              f"loss={metrics['loss_total']:.5f}  "
              f"alpha={metrics['loss_alpha']:.5f}  "
              f"color={metrics['loss_color']:.5f}  "
              f"pool={dist}")
 
print("\nTraining complete.")

In [ ]:
nca_cfg = NCAConfig(
    grid_size                 = tuple(config["grid"]["size"]),
    hidden_channels           = config["cell"]["hidden_channels"],
    visible_channels          = config["cell"]["visible_channels"],
    alive_threshold           = config["cell"]["alive_threshold"],
    task_channels             = config["cell"]["task_channels"],
    pos_channels              = config["cell"]["pos_channels"],
    perception_kernel_radius  = config["perception"]["kernel_radius"],
    perception_channel_groups = config["perception"]["channel_groups"],
    update_hidden_dim         = config["update"]["hidden_dim"],
    update_stochastic         = config["update"]["stochastic_update"],
    update_fire_rate          = config["update"]["fire_rate"],
)
nca_model = NCAModel(nca_cfg)
nca_model.grid.load_state_dict(model.state_dict())
 
save_path = f"{DRIVE_DIR}/model_final_run{logger.run_id}.pt"
nca_model.save(save_path)
print(f"NCAModel saved -> {save_path}")

In [ ]:
df = pd.read_csv(logger._loss_path)
 
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.semilogy(df["epoch"], df["loss_alpha"],    lw=0.8, color="#1f77b4", label="alpha loss")
ax.semilogy(df["epoch"], df["loss_color"],    lw=0.8, color="#ff7f0e", label="color loss")
ax.semilogy(df["epoch"], df["loss_overflow"], lw=0.8, color="#2ca02c", label="overflow loss")
ax.semilogy(df["epoch"], df["loss_total"],    lw=1.4, color="#d62728", label="total loss")
 
for ep, label in [(PHASE_EPOCHS, "task 0+1 joint"),
                  (PHASE_EPOCHS * 2, "task 0+1+2 joint")]:
    ax.axvline(ep, color="grey", lw=1, ls="--", alpha=0.7)
    ax.text(ep + 40, 0.92, label,
            transform=ax.get_xaxis_transform(),
            color="grey", fontsize=8, va="top")
 
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")
ax.set_title(f"Curriculum joint training -- 3D-NCA  "
             f"[run {logger.run_id}, {N_TASKS} tasks, {TOTAL_EPOCHS} epochs]")
ax.legend(loc="upper right", fontsize=8, framealpha=0.7)
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.savefig(f"{logger.run_dir}/loss_curve.png", dpi=150)
plt.show()

In [ ]:
SNAP_AT = [0, 8, 24, 48, 80, 120, 160]
 
def run_with_snaps(state, snap_at):
    snaps, done = [], 0
    with torch.no_grad():
        for target_step in snap_at:
            n = target_step - done
            if n > 0:
                state = model(state, steps=n, use_checkpointing=False)
                state = torch.clamp(state, -1.0, 1.0)
                done  = target_step
            snaps.append(state.detach().cpu())
    return state.detach(), snaps
 
model.eval()
 
state = new_seed(0)
state, snaps_tetra  = run_with_snaps(state, SNAP_AT)
 
state = relabel(state, 1)
state, snaps_cube   = run_with_snaps(state, SNAP_AT)
 
state = relabel(state, 2)
state, snaps_sphere = run_with_snaps(state, SNAP_AT)
 
all_phase_snaps  = [snaps_tetra, snaps_cube, snaps_sphere]
PHASE_NAMES      = ["Seed -> Tetrahedron", "Tetrahedron -> Cube", "Cube -> Sphere"]
 
print("Inference alive voxels at final step:")
for name, snaps in zip(TASK_NAMES, all_phase_snaps):
    alive = int((snaps[-1][0, -1] > cell_cfg.alive_threshold).sum().item())
    print(f"  {name:12s}: {alive:>5}")
 

In [ ]:
N_SNAPS = len(SNAP_AT)
fig = plt.figure(figsize=(N_SNAPS * 2.2, N_TASKS * 2.6))
gs  = gridspec.GridSpec(N_TASKS, N_SNAPS, figure=fig, wspace=0.04, hspace=0.30)
 
for row, (phase_name, snaps) in enumerate(zip(PHASE_NAMES, all_phase_snaps)):
    for col, (step, snap) in enumerate(zip(SNAP_AT, snaps)):
        ax = fig.add_subplot(gs[row, col], projection="3d")
        show_state_rgba_mpl(snap, visible_channels=VIS,
                            threshold=0.20, surface_only=True,
                            point_size=7, view_angle=(28, 40),
                            ax=ax, show=False, title="")
        ax.set_title(f"+{step}", fontsize=7, pad=1)
        if col == 0:
            ax.text2D(-0.22, 0.5, phase_name,
                      transform=ax.transAxes, fontsize=8, fontweight="bold",
                      va="center", ha="right", rotation=90)
 
fig.suptitle(
    f"Metamorphogenesis: Tetrahedron -> Cube -> Sphere  [run {logger.run_id}]\n"
    "Each row continues from the previous row's final state -- no re-seeding.",
    fontsize=10, y=1.02,
)
plt.tight_layout()
out = f"{logger.run_dir}/morph_progression.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {out}")
 

In [ ]:
fig, axes = plt.subplots(2, N_TASKS, figsize=(12, 8),
                         subplot_kw={"projection": "3d"})
 
for col, (name, tgt_rgba, snaps) in enumerate(
        zip(TASK_NAMES, TARGETS, all_phase_snaps)):
 
    n_t = show_volume_rgba_mpl(tgt_rgba, ax=axes[0, col],
                               surface_only=True, threshold=0.15,
                               point_size=14, view_angle=(28, 40),
                               show=False, title="")
    axes[0, col].set_title(f"Target\n{name}  ({n_t} vx)", fontsize=9)
 
    n_s = show_state_rgba_mpl(snaps[-1], visible_channels=VIS, ax=axes[1, col],
                              surface_only=True, threshold=0.20,
                              point_size=14, view_angle=(28, 40),
                              show=False, title="")
    axes[1, col].set_title(f"Output\n{name}  ({n_s} vx)", fontsize=9)
 
for row, label in enumerate(["Target", "Model output"]):
    axes[row, 0].text2D(-0.18, 0.5, label,
                        transform=axes[row, 0].transAxes,
                        fontsize=10, fontweight="bold",
                        va="center", ha="right", rotation=90)
 
fig.suptitle(f"Target vs model output  [run {logger.run_id}]",
             fontsize=12, y=1.01)
plt.tight_layout()
out = f"{logger.run_dir}/comparison.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {out}")
 
print(f"\n-- Done --------------------------------------------------")
print(f"  Run dir : {logger.run_dir}")
print(f"  Model   : {save_path}")

In [ ]:
!zip -r /kaggle/working/morph_final.zip /kaggle/working/nca_morph